# OpenSetDGA — Re-run LGBM Energy Fix

Chạy lại **lgbm_binary** và **lgbm_multi** sau khi fix energy score:
- **Bug cũ:** `energy_score(proba)` → `logsumexp(log p) = log(Σp) = 0` khi T=1 → constant score
- **Fix:** dùng raw logits từ `booster_.predict(X, raw_score=True)` theo Liu et al. 2020

**Seeds:** 1, 7, 42, 99, 314  
**Device:** CPU  
**Ước tính:** ~30–40 phút (10 model × ~3–4 phút/model)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'lightgbm', 'huggingface_hub', 'tldextract'], check=True)
print('Packages ready.')

In [ ]:
import os
from pathlib import Path

WORKDIR = Path('/kaggle/working/OpenSetDGA-Detection')

if not WORKDIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/quanturong/OpenSetDGA-Detection.git',
         str(WORKDIR)],
        check=True
    )
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', str(WORKDIR), 'pull'], check=False)
    print('Repo already exists — pulled latest.')

os.chdir(WORKDIR)

# Verify energy fix is present
src = (WORKDIR / 'src' / 'train_baseline.py').read_text(encoding='utf-8')
assert 'raw_logit' in src and 'booster_.predict' in src, \
    'Fix not found in train_baseline.py — check git pull'

src_mc = (WORKDIR / 'src' / 'train_multiclass.py').read_text(encoding='utf-8')
assert 'raw_logits' in src_mc and 'raw_score=True' in src_mc, \
    'Fix not found in train_multiclass.py — check git pull'

print('Fix verified in both train_baseline.py and train_multiclass.py.')
print(f'Working dir: {os.getcwd()}')

In [ ]:
from huggingface_hub import snapshot_download
import pandas as pd

DATA_DIR = WORKDIR / 'data' / 'processed'

if not (DATA_DIR / 'known' / 'train.csv').exists():
    print('Downloading dataset...')
    snapshot_download(
        repo_id='ThanhPhuongtphz/OpenSetDGA-Detection',
        repo_type='dataset',
        local_dir=str(DATA_DIR),
        ignore_patterns=['*.git*', '*.md', '*.txt'],
    )
    print('Download complete.')
else:
    print('Data already present.')

for p in [
    DATA_DIR / 'known' / 'train.csv',
    DATA_DIR / 'known' / 'test_known.csv',
    DATA_DIR / 'unknown_family' / 'test_unknown_family.csv',
    DATA_DIR / 'unknown_ood' / 'test_unknown_ood.csv',
]:
    n = len(pd.read_csv(p))
    print(f'  OK  {p.name}  ({n:,} rows)')

In [ ]:
SEEDS = [1, 7, 42, 99, 314]

print('=== Running lgbm_binary (5 seeds) ===')
result = subprocess.run(
    [sys.executable, 'src/run_multi_seed.py',
     '--seeds'] + [str(s) for s in SEEDS] + [
     '--only', 'lgbm_binary',
     '--device', 'cpu',
     '--run_dir', 'data/processed'],
    cwd=str(WORKDIR)
)
print(f'Exit code: {result.returncode}')

In [ ]:
print('=== Running lgbm_multi (5 seeds) ===')
result = subprocess.run(
    [sys.executable, 'src/run_multi_seed.py',
     '--seeds'] + [str(s) for s in SEEDS] + [
     '--only', 'lgbm_multi',
     '--device', 'cpu',
     '--run_dir', 'data/processed'],
    cwd=str(WORKDIR)
)
print(f'Exit code: {result.returncode}')

In [ ]:
import json, numpy as np

SPLITS   = ['unknown_family', 'unknown_ood']
SCORERS  = ['msp', 'energy']
MODELS   = ['lgbm_binary', 'lgbm_multi']

def load(model, seed):
    p = WORKDIR / 'baseline_out' / f'{model}_s{seed}' / 'results.json'
    return json.load(open(p)) if p.exists() else {}

def stats(vals):
    vals = [v for v in vals if v is not None]
    if not vals: return None, None
    m = np.mean(vals); s = np.std(vals)
    return m, s

print(f'{'Model/Scorer/Split':<35} {"AUROC":>8} {"AUPR":>8} {"FPR@95":>8} {"Prec@95":>9}')
print('-' * 75)

for model in MODELS:
    for scorer in SCORERS:
        for split in SPLITS:
            key = f'ood_{scorer}_{split}'
            aurocs, auprs, fprs, precs = [], [], [], []
            missing = []
            for seed in SEEDS:
                d = load(model, seed).get(key, {})
                if d:
                    aurocs.append(d['auroc'])
                    auprs.append(d['aupr_out'])
                    fprs.append(d['fpr_at_tpr'])
                    precs.append(d.get('precision_at_tpr'))
                else:
                    missing.append(seed)

            if missing:
                print(f'  MISSING seeds {missing} for {model}/{scorer}/{split}')
                continue

            ma, _ = stats(aurocs)
            mp, _ = stats(auprs)
            mf, _ = stats(fprs)
            mc, _ = stats(precs)
            label = f'{model}/{scorer}/{split}'
            pc_str = f'{mc:.4f}' if mc is not None else '—'
            print(f'{label:<35} {ma:>8.4f} {mp:>8.4f} {mf:>8.4f} {pc_str:>9}')

print()
# Reference: base rate for constant-score classifier
n_id = len(pd.read_csv(DATA_DIR / 'known' / 'test_known.csv'))
n_uf = len(pd.read_csv(DATA_DIR / 'unknown_family' / 'test_unknown_family.csv'))
n_oo = len(pd.read_csv(DATA_DIR / 'unknown_ood' / 'test_unknown_ood.csv'))
print(f'Base rate uf = {n_uf/(n_uf+n_id):.4f}  |  Base rate oo = {n_oo/(n_oo+n_id):.4f}')
print('(Energy AUPR should now be > base rate if fix works correctly)')

In [ ]:
# Per-seed detail for energy rows
print('Per-seed AUPR (energy only):')
for model in MODELS:
    for split in SPLITS:
        key = f'ood_energy_{split}'
        vals = [load(model, s).get(key, {}).get('aupr_out') for s in SEEDS]
        vals_str = '  '.join(f'{v:.4f}' if v is not None else '—' for v in vals)
        m = np.mean([v for v in vals if v is not None])
        print(f'  {model}/{split}: {vals_str}  → mean={m:.4f}')

In [ ]:
import zipfile

zip_path = Path('/kaggle/working/lgbm_energy_fix.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for model in MODELS:
        for seed in SEEDS:
            d = WORKDIR / 'baseline_out' / f'{model}_s{seed}'
            if not d.exists():
                print(f'WARNING: missing {model}_s{seed}')
                continue
            for f in d.rglob('*'):
                if f.is_file() and f.suffix in ('.json', '.txt', '.csv'):
                    zf.write(f, f'{model}_s{seed}/{f.name}')

size_kb = zip_path.stat().st_size / 1024
print(f'Saved: {zip_path.name}  ({size_kb:.1f} KB)')
print('Download từ Output tab.')